# Model testing

Load data and convert

In [2]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Set the number of threads for parallel processing to the number of CPU cores
n_cores = os.cpu_count() or 1
# Use half of the available cores to avoid overheating
n_cores = max(1, n_cores // 2)
os.environ["OMP_NUM_THREADS"] = str(n_cores)
print(f"Using {n_cores} threads for parallel processing.")

# Load data
TRAIN_DATA_PATH = Path("data/train_data.parquet")
TEST_DATA_PATH = Path("data/test_data.parquet")
train_df = pd.read_parquet(TRAIN_DATA_PATH)
test_df = pd.read_parquet(TEST_DATA_PATH)

# Convert indices to numeric values
train_df.index = pd.to_numeric(train_df.index)
test_df.index = pd.to_numeric(test_df.index)

# Drop datetime features
cols_to_drop = ["issue_d", "earliest_cr_line"]
train_df = train_df.drop(columns=cols_to_drop)
test_df = test_df.drop(columns=cols_to_drop)

Using 4 threads for parallel processing.


In [3]:
# Split data into features and targets
train_full_X = train_df.drop(["target", "target_annual_roi"], axis=1)
train_full_y_cat = train_df["target"].astype(int)
train_full_y_reg = train_df["target_annual_roi"]

test_full_X = test_df.drop(["target", "target_annual_roi"], axis=1)
test_full_y_cat = test_df["target"].astype(int)
test_full_y_reg = test_df["target_annual_roi"]

In [4]:
# Create subsets of the training data for different sizes
train_1k_X = train_full_X.tail(1000)
train_1k_y_cat = train_full_y_cat.tail(1000)
train_1k_y_reg = train_full_y_reg.tail(1000)

train_10k_X = train_full_X.tail(10000)
train_10k_y_cat = train_full_y_cat.tail(10000)
train_10k_y_reg = train_full_y_reg.tail(10000)

train_100k_X = train_full_X.tail(100000)
train_100k_y_cat = train_full_y_cat.tail(100000)
train_100k_y_reg = train_full_y_reg.tail(100000)

In [5]:
# Function to get the number of trees in a fitted model based on its type
def get_number_of_trees(model_name, fitted_model):
    if model_name == "XGBoost":
        return fitted_model.get_booster().num_boosted_rounds()
    if model_name == "LightGBM":
        return fitted_model.booster_.num_trees()
    if model_name == "CatBoost":
        return fitted_model.tree_count_
    if model_name in ["NGBoost", "GBM"]:
        return fitted_model.n_estimators
    if model_name in ["HistGBM", "PGBM"]:
        return fitted_model.max_iter
    return 1

## Classification

Preprocessors

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

# Print categorical columns
cols_to_nominal_cat = train_df.select_dtypes(
    include=["object", "category"]
).columns.tolist()
print("Categorical columns:")
for col in cols_to_nominal_cat:
    print(f"- {col}")

# Set maximal cardinality for one-hot encoding
cardinality = train_df[cols_to_nominal_cat].nunique()
threshold_for_ohe = 5
cols_for_ohe = cardinality[cardinality <= threshold_for_ohe].index.tolist()
cols_for_te = cardinality[cardinality > threshold_for_ohe].index.tolist()
ohe_categories = []
for col in cols_for_ohe:
    unique_cats = train_df[col].dropna().unique().tolist()
    ohe_categories.append(unique_cats)

# Create OneHotEncoder
ohe_transformer = OneHotEncoder(
    categories=ohe_categories,
    drop="if_binary",
    handle_unknown="ignore",
    sparse_output=False,
)

# Create TargetEncoder for rest of nominal categorical features
target_transformer_nominal = TargetEncoder(target_type="binary", smooth="auto")

# Create whole preprocessor for models which are unable to handle categorical
# features but able to handle missing values
numeric_preprocessor_cat = ColumnTransformer(
    transformers=[
        ("ohe", ohe_transformer, cols_for_ohe),
        ("target_enc", target_transformer_nominal, cols_for_te),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

# Create whole preprocessor for GBM with imputation based on hyperparameter tuning
imputed_numeric_preprocessor_1k_cat = make_pipeline(
    numeric_preprocessor_cat, SimpleImputer(strategy="mean")
).set_output(transform="pandas")

imputed_numeric_preprocessor_10k_cat = make_pipeline(
    numeric_preprocessor_cat, SimpleImputer(strategy="median")
).set_output(transform="pandas")

imputed_numeric_preprocessor_100k_cat = make_pipeline(
    numeric_preprocessor_cat, SimpleImputer(strategy="mean")
).set_output(transform="pandas")

# Create passthrough preprocessor for models which are able to handle
# categorical features and missing values
passthrough_preprocessor = "passthrough"

# Create preprocessor for CatBoost filling missing values of categorical
# features with "Missing"
catboost_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat_imputer",
            SimpleImputer(strategy="constant", fill_value="Missing"),
            cols_to_nominal_cat,
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

Categorical columns:
- home_ownership
- verification_status
- purpose
- addr_state
- initial_list_status
- application_type
- disbursement_method


### Model definition

In [7]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from ngboost import NGBClassifier
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.tree import DecisionTreeRegressor
from ngboost.distns import Bernoulli

# Dictionary of models without hyperparameter tuning, with appropriate preprocessors
models_no_tuning_cat = {
    "XGBoost": (
        passthrough_preprocessor,
        XGBClassifier(random_state=42, enable_categorical=True),
    ),
    "LightGBM": (passthrough_preprocessor, LGBMClassifier(random_state=42, verbose=-1)),
    "CatBoost": (
        catboost_preprocessor,
        CatBoostClassifier(
            random_state=42, verbose=0, cat_features=cols_to_nominal_cat
        ),
    ),
    "NGBoost": (numeric_preprocessor_cat, NGBClassifier(random_state=42, verbose=False)),
    "GBM": (
        imputed_numeric_preprocessor_1k_cat,
        GradientBoostingClassifier(random_state=42),
    ),
    "HistGBM": (numeric_preprocessor_cat, HistGradientBoostingClassifier(random_state=42)),
    "Dummy - Most Frequent": (numeric_preprocessor_cat, DummyClassifier(strategy="prior")),
}

In [8]:
# Dictionaries of optimal hyperparameters for 1k dataset
xgboost_1k_cat_params = {
    "booster": "gbtree",
    "learning_rate": 0.049961658713390054,
    "min_split_loss": 3.1792890444034416,
    "max_depth": 4,
    "min_child_weight": 0.03446885315674068,
    "max_delta_step": 8.818441265852027,
    "colsample_bytree": 0.6612602163953242,
    "colsample_bylevel": 0.4402275230048245,
    "colsample_bynode": 0.44195112565215633,
    "reg_lambda": 3.821278757725177,
    "reg_alpha": 5.339764624690546,
    "scale_pos_weight": 6.723400891593152,
    "grow_policy": "depthwise",
    "max_leaves": 23,
    "max_bin": 247,
    "max_cat_to_onehot": 8,
    "max_cat_threshold": 26,
    "n_estimators": 614,
    "sampling_method": "uniform",
    "subsample": 0.6152248546699406,
    "random_state": 42,
    "n_jobs": n_cores,
    "verbosity": 0,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "enable_categorical": True,
}
lightgbm_1k_cat_params = {
    "boosting_type": "gbdt",
    "num_leaves": 19,
    "max_depth": 6,
    "learning_rate": 0.013118621303382356,
    "scale_pos_weight": 4.234357177755502,
    "min_split_gain": 4.150101489297818,
    "min_child_weight": 0.1439268407905771,
    "min_child_samples": 47,
    "colsample_bytree": 0.6376678823246574,
    "reg_alpha": 0.29085902006459885,
    "reg_lambda": 0.3198616601646805,
    "colsample_bynode": 0.4286302115954977,
    "min_data_per_group": 97,
    "max_cat_threshold": 8,
    "cat_l2": 30.200602897458936,
    "cat_smooth": 0.014021044837121193,
    "max_cat_to_onehot": 6,
    "max_bin": 166,
    "n_estimators": 372,
    "subsample": 0.6440572461920292,
    "subsample_freq": 4,
    "random_state": 42,
    "n_jobs": n_cores,
    "verbose": -1,
    "objective": "binary",
    "metric": "auc",
}
catboost_1k_cat_params = {
    "iterations": 433,
    "learning_rate": 0.03554345124418867,
    "depth": 4,
    "l2_leaf_reg": 9.840089950178873,
    "random_strength": 0.02805738171539979,
    "rsm": 0.9135819869398158,
    "border_count": 64,
    "scale_pos_weight": 6.980459859158197,
    "boosting_type": "Ordered",
    "bootstrap_type": "Bayesian",
    "score_function": "L2",
    "bagging_temperature": 9.907283731079449,
    "random_state": 42,
    "thread_count": n_cores,
    "verbose": False,
    "objective": "Logloss",
    "cat_features": [
        "home_ownership",
        "verification_status",
        "purpose",
        "addr_state",
        "initial_list_status",
        "application_type",
        "disbursement_method",
    ],
}
ngboost_1k_cat_params = {
    "n_estimators": 737,
    "learning_rate": 0.022671452328580047,
    "minibatch_frac": 0.8921512142357929,
    "col_sample": 0.4412224129149527,
    "random_state": 42,
    "verbose": False,
}
gbm_1k_cat_params = {
    "n_estimators": 579,
    "learning_rate": 0.11372034355457401,
    "max_depth": 3,
    "subsample": 0.6608071029790215,
    "min_samples_split": 44,
    "min_samples_leaf": 36,
    "min_weight_fraction_leaf": 0.13651164177932174,
    "min_impurity_decrease": 0.91953008742171,
    "max_features": "log2",
    "max_leaf_nodes": 22,
    "ccp_alpha": 2.1238742124806448e-05,
    "random_state": 42,
    "verbose": 0,
}
histgbm_1k_cat_params = {
    "max_iter": 809,
    "learning_rate": 0.005855268047892045,
    "max_depth": 6,
    "min_samples_leaf": 73,
    "max_features": 0.6326799701936273,
    "max_leaf_nodes": 31,
    "l2_regularization": 0.00047402616662400567,
    "max_bins": 252,
    "random_state": 42,
    "verbose": 0,
}
ngboost_1k_cat_tree_regressor_params = {
    "criterion": "friedman_mse",
    "max_depth": 7,
    "min_samples_split": 61,
    "min_samples_leaf": 66,
    "min_weight_fraction_leaf": 0.1010094827157183,
    "max_features": "sqrt",
    "ccp_alpha": 1.8200306725360187e-05,
    "max_leaf_nodes": 25,
    "min_impurity_decrease": 0.05036939294247465,
    "random_state": 42,
}
ngboost_1k_cat_tree_regressor = DecisionTreeRegressor(
    **ngboost_1k_cat_tree_regressor_params
)

# Dictionary of models with hyperparameter tuning for 1k dataset with preprocessors
models_tuned_1k_cat = {
    "XGBoost": (passthrough_preprocessor, XGBClassifier(**xgboost_1k_cat_params)),
    "LightGBM": (passthrough_preprocessor, LGBMClassifier(**lightgbm_1k_cat_params)),
    "CatBoost": (
        catboost_preprocessor,
        CatBoostClassifier(**catboost_1k_cat_params),
    ),
    "NGBoost": (
        numeric_preprocessor_cat,
        NGBClassifier(
            **ngboost_1k_cat_params, Dist=Bernoulli, Base=ngboost_1k_cat_tree_regressor
        ),
    ),
    "GBM": (
        imputed_numeric_preprocessor_1k_cat,
        GradientBoostingClassifier(**gbm_1k_cat_params),
    ),
    "HistGBM": (
        numeric_preprocessor_cat,
        HistGradientBoostingClassifier(**histgbm_1k_cat_params),
    ),
    "Dummy - Most Frequent": (numeric_preprocessor_cat, DummyClassifier(strategy="prior"))
}

In [9]:
# Dictionaries of optimal hyperparameters for 10k dataset
xgboost_10k_cat_params = {
    "booster": "gbtree",
    "learning_rate": 0.002660634878062164,
    "min_split_loss": 1.5564766508977637,
    "max_depth": 3,
    "min_child_weight": 0.002666795494419774,
    "max_delta_step": 3.237106539752366,
    "colsample_bytree": 0.6334129748685073,
    "colsample_bylevel": 0.31232752047853835,
    "colsample_bynode": 0.5663544691691175,
    "reg_lambda": 0.866361380309008,
    "reg_alpha": 0.34097661390173023,
    "scale_pos_weight": 2.237901939773783,
    "grow_policy": "lossguide",
    "max_leaves": 14,
    "max_bin": 203,
    "max_cat_to_onehot": 2,
    "max_cat_threshold": 4,
    "n_estimators": 1828,
    "sampling_method": "gradient_based",
    "subsample": 0.21905666352297487,
    "random_state": 42,
    "n_jobs": n_cores,
    "verbosity": 0,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "enable_categorical": True,
}
lightgbm_10k_cat_params = {
    "boosting_type": "goss",
    "num_leaves": 26,
    "max_depth": 8,
    "learning_rate": 0.0057452620856591865,
    "scale_pos_weight": 2.5548028672067264,
    "min_split_gain": 8.402184412779064,
    "min_child_weight": 0.7889913973072552,
    "min_child_samples": 67,
    "colsample_bytree": 0.3322094184323836,
    "reg_alpha": 0.009195021998688829,
    "reg_lambda": 1.241109986317591,
    "colsample_bynode": 0.3432164431126759,
    "min_data_per_group": 13,
    "max_cat_threshold": 3,
    "cat_l2": 4.82194592281236,
    "cat_smooth": 0.41553169506659904,
    "max_cat_to_onehot": 1,
    "max_bin": 66,
    "n_estimators": 1022,
    "top_rate": 0.07974625682135272,
    "other_rate": 0.15560637534232696,
    "random_state": 42,
    "n_jobs": n_cores,
    "verbose": -1,
    "objective": "binary",
    "metric": "auc",
}
catboost_10k_cat_params = {
    "iterations": 1004,
    "learning_rate": 0.007488378628866546,
    "depth": 8,
    "l2_leaf_reg": 0.008738906090086759,
    "random_strength": 0.6403151618824401,
    "rsm": 0.7659083035388369,
    "border_count": 162,
    "scale_pos_weight": 5.959581256258957,
    "boosting_type": "Plain",
    "bootstrap_type": "Bayesian",
    "grow_policy": "Lossguide",
    "max_leaves": 17,
    "min_data_in_leaf": 91,
    "bagging_temperature": 8.58954612632319,
    "random_state": 42,
    "thread_count": n_cores,
    "verbose": False,
    "objective": "Logloss",
    "cat_features": [
        "home_ownership",
        "verification_status",
        "purpose",
        "addr_state",
        "initial_list_status",
        "application_type",
        "disbursement_method",
    ],
}
ngboost_10k_cat_params = {
    "n_estimators": 732,
    "learning_rate": 0.004987425762907027,
    "minibatch_frac": 0.7943140099019186,
    "col_sample": 0.7946835959334093,
    "random_state": 42,
    "verbose": False,
}
gbm_10k_cat_params = {
    "n_estimators": 1588,
    "learning_rate": 0.00549390058546573,
    "max_depth": 3,
    "subsample": 0.634594524243599,
    "min_samples_split": 27,
    "min_samples_leaf": 79,
    "min_weight_fraction_leaf": 0.002500791243609685,
    "min_impurity_decrease": 0.5395017070575661,
    "max_features": "log2",
    "max_leaf_nodes": 8,
    "ccp_alpha": 2.565584578912088e-05,
    "random_state": 42,
    "verbose": 0,
}
histgbm_10k_cat_params = {
    "max_iter": 1352,
    "learning_rate": 0.0047997605390942095,
    "max_depth": 4,
    "min_samples_leaf": 53,
    "max_features": 0.2772575122505353,
    "max_leaf_nodes": 26,
    "l2_regularization": 93.90576777776629,
    "max_bins": 162,
    "random_state": 42,
    "verbose": 0,
}
ngboost_10k_cat_tree_regressor_params = {
    "criterion": "friedman_mse",
    "max_depth": 4,
    "min_samples_split": 22,
    "min_samples_leaf": 17,
    "min_weight_fraction_leaf": 0.02731981862937974,
    "max_features": "log2",
    "ccp_alpha": 2.0144676573632977e-05,
    "max_leaf_nodes": 28,
    "min_impurity_decrease": 0.18904335829713417,
    "random_state": 42,
}
ngboost_10k_cat_tree_regressor = DecisionTreeRegressor(
    **ngboost_10k_cat_tree_regressor_params
)

# Dictionary of models with hyperparameter tuning for 10k dataset with preprocessors
models_tuned_10k_cat = {
    "XGBoost": (passthrough_preprocessor, XGBClassifier(**xgboost_10k_cat_params)),
    "LightGBM": (passthrough_preprocessor, LGBMClassifier(**lightgbm_10k_cat_params)),
    "CatBoost": (
        catboost_preprocessor,
        CatBoostClassifier(**catboost_10k_cat_params),
    ),
    "NGBoost": (
        numeric_preprocessor_cat,
        NGBClassifier(
            **ngboost_10k_cat_params,
            Dist=Bernoulli,
            Base=ngboost_10k_cat_tree_regressor,
        ),
    ),
    "GBM": (
        imputed_numeric_preprocessor_10k_cat,
        GradientBoostingClassifier(**gbm_10k_cat_params),
    ),
    "HistGBM": (
        numeric_preprocessor_cat,
        HistGradientBoostingClassifier(**histgbm_10k_cat_params),
    ),
    "Dummy - Most Frequent": (numeric_preprocessor_cat, DummyClassifier(strategy="prior")),
}

In [10]:
# Dictionaries of optimal hyperparameters for 100k dataset
xgboost_100k_cat_params = {
    "booster": "gbtree",
    "learning_rate": 0.0009878389008570133,
    "min_split_loss": 1.6211471225466279,
    "max_depth": 18,
    "min_child_weight": 6.506529198811776,
    "max_delta_step": 4.996596322660088,
    "colsample_bytree": 0.5497212085853451,
    "colsample_bylevel": 0.6402797842177399,
    "colsample_bynode": 0.4945766078267192,
    "reg_lambda": 3.901817763099591e-08,
    "reg_alpha": 9.955075730252275,
    "scale_pos_weight": 2.385489120852272,
    "grow_policy": "lossguide",
    "max_leaves": 70,
    "max_bin": 428,
    "max_cat_to_onehot": 10,
    "max_cat_threshold": 686,
    "n_estimators": 9010,
    "sampling_method": "uniform",
    "subsample": 0.34315380106756327,
    "random_state": 42,
    "n_jobs": n_cores,
    "verbosity": 0,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "enable_categorical": True,
}
lightgbm_100k_cat_params = {
    "boosting_type": "goss",
    "num_leaves": 444,
    "max_depth": 15,
    "learning_rate": 0.0013490712742580085,
    "scale_pos_weight": 2.529589934768151,
    "min_split_gain": 3.4510864669147536,
    "min_child_weight": 0.19239271728859866,
    "min_child_samples": 234,
    "colsample_bytree": 0.31324120666189326,
    "reg_alpha": 0.6545836243558268,
    "reg_lambda": 0.1069421692665621,
    "colsample_bynode": 0.21348019642019067,
    "min_data_per_group": 803,
    "max_cat_threshold": 819,
    "cat_l2": 0.00011204200863439848,
    "cat_smooth": 1.5621591746591863,
    "max_cat_to_onehot": 36,
    "max_bin": 258,
    "n_estimators": 8720,
    "top_rate": 0.24249410683009498,
    "other_rate": 0.42369427528780756,
    "random_state": 42,
    "n_jobs": n_cores,
    "verbose": -1,
    "objective": "binary",
    "metric": "auc",
}
catboost_100k_cat_params = {
    "iterations": 8940,
    "learning_rate": 0.0025302116751832124,
    "depth": 8,
    "l2_leaf_reg": 98.54551873139987,
    "random_strength": 0.030339567093011274,
    "rsm": 0.8587177152995968,
    "border_count": 185,
    "scale_pos_weight": 6.177887840704495,
    "boosting_type": "Ordered",
    "bootstrap_type": "Bernoulli",
    "score_function": "L2",
    "subsample": 0.7842935885515135,
    "random_state": 42,
    "thread_count": n_cores,
    "verbose": False,
    "objective": "Logloss",
    "cat_features": [
        "home_ownership",
        "verification_status",
        "purpose",
        "addr_state",
        "initial_list_status",
        "application_type",
        "disbursement_method",
    ],
}
ngboost_100k_cat_params = {
    "n_estimators": 9100,
    "learning_rate": 0.001001318427544568,
    "minibatch_frac": 0.6270741484049653,
    "col_sample": 0.5670544735298176,
    "random_state": 42,
    "verbose": False,
}
gbm_100k_cat_params = {
    "n_estimators": 4750,
    "learning_rate": 0.01648269774352038,
    "max_depth": 16,
    "subsample": 0.9090892401385037,
    "min_samples_split": 443,
    "min_samples_leaf": 75,
    "min_weight_fraction_leaf": 0.09091249242263949,
    "min_impurity_decrease": 0.6068460342082242,
    "max_features": None,
    "max_leaf_nodes": 82,
    "ccp_alpha": 1.9782964485238465e-05,
    "random_state": 42,
    "verbose": 0,
}
histgbm_100k_cat_params = {
    "max_iter": 3930,
    "learning_rate": 0.0028610506052507275,
    "max_depth": 18,
    "min_samples_leaf": 491,
    "max_features": 0.25844075959233276,
    "max_leaf_nodes": 273,
    "l2_regularization": 1.2200390997850423,
    "max_bins": 219,
    "random_state": 42,
    "verbose": 0,
}
ngboost_100k_cat_tree_regressor_params = {
    "criterion": "friedman_mse",
    "max_depth": 20,
    "min_samples_split": 31,
    "min_samples_leaf": 474,
    "min_weight_fraction_leaf": 0.0021335723044131113,
    "max_features": "sqrt",
    "ccp_alpha": 0.00028039529173976675,
    "max_leaf_nodes": 300,
    "min_impurity_decrease": 0.29132470764416524,
    "random_state": 42,
}
ngboost_100k_cat_tree_regressor = DecisionTreeRegressor(
    **ngboost_100k_cat_tree_regressor_params
)

# Dictionary of models with hyperparameter tuning for 100k dataset with preprocessors
models_tuned_100k_cat = {
    "XGBoost": (passthrough_preprocessor, XGBClassifier(**xgboost_100k_cat_params)),
    "LightGBM": (passthrough_preprocessor, LGBMClassifier(**lightgbm_100k_cat_params)),
    "CatBoost": (
        catboost_preprocessor,
        CatBoostClassifier(**catboost_100k_cat_params),
    ),
    "NGBoost": (
        numeric_preprocessor_cat,
        NGBClassifier(
            **ngboost_100k_cat_params,
            Dist=Bernoulli,
            Base=ngboost_100k_cat_tree_regressor,
        ),
    ),
    "GBM": (
        imputed_numeric_preprocessor_100k_cat,
        GradientBoostingClassifier(**gbm_100k_cat_params),
    ),
    "HistGBM": (
        numeric_preprocessor_cat,
        HistGradientBoostingClassifier(**histgbm_100k_cat_params),
    ),
    "Dummy - Most Frequent": (numeric_preprocessor_cat, DummyClassifier(strategy="prior")),
}

In [11]:
# Dictionaries of optimal hyperparameters for full dataset
xgboost_full_cat_params = {
    "booster": "gbtree",
    "learning_rate": 0.0032927768001972434,
    "min_split_loss": 8.210752775090127,
    "max_depth": 7,
    "min_child_weight": 0.15538873240656356,
    "max_delta_step": 0.27402281636668574,
    "colsample_bytree": 0.49786443299241995,
    "colsample_bylevel": 0.5536721341549702,
    "colsample_bynode": 0.6937873385849318,
    "reg_lambda": 5.867840852070052,
    "reg_alpha": 0.00036494609794361513,
    "scale_pos_weight": 2.197402352020041,
    "grow_policy": "depthwise",
    "max_leaves": 213,
    "max_bin": 81,
    "max_cat_to_onehot": 2,
    "max_cat_threshold": 31,
    "n_estimators": 9670,
    "sampling_method": "uniform",
    "subsample": 0.693420186711674,
    "random_state": 42,
    "n_jobs": n_cores,
    "verbosity": 0,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "enable_categorical": True,
}
lightgbm_full_cat_params = {
    "boosting_type": "gbdt",
    "num_leaves": 176,
    "max_depth": 18,
    "learning_rate": 0.0013263394883988124,
    "scale_pos_weight": 5.720161074894982,
    "min_split_gain": 7.015394333142449,
    "min_child_weight": 0.7877329829167548,
    "min_child_samples": 287,
    "colsample_bytree": 0.6513068572006349,
    "reg_alpha": 0.25347089784492205,
    "reg_lambda": 5.51570221223381e-05,
    "colsample_bynode": 0.33764943797808045,
    "min_data_per_group": 550,
    "max_cat_threshold": 187,
    "cat_l2": 2.2848847335233635e-07,
    "cat_smooth": 1.2867859076903958e-05,
    "max_cat_to_onehot": 46,
    "max_bin": 127,
    "n_estimators": 9670,
    "subsample": 0.7854117062367028,
    "subsample_freq": 10,
    "random_state": 42,
    "n_jobs": n_cores,
    "verbose": -1,
    "objective": "binary",
    "metric": "auc",
}
catboost_full_cat_params = {
    "iterations": 8620,
    "learning_rate": 0.005036874795758855,
    "depth": 7,
    "l2_leaf_reg": 28.934704951911797,
    "random_strength": 0.5144245877609427,
    "rsm": 0.5706570280810435,
    "border_count": 105,
    "scale_pos_weight": 7.201993563101878,
    "boosting_type": "Ordered",
    "bootstrap_type": "Bernoulli",
    "score_function": "L2",
    "subsample": 0.5675364012486861,
    "random_state": 42,
    "thread_count": n_cores,
    "verbose": False,
    "objective": "Logloss",
    "cat_features": [
        "home_ownership",
        "verification_status",
        "purpose",
        "addr_state",
        "initial_list_status",
        "application_type",
        "disbursement_method",
    ],
}
histgbm_full_cat_params = {
    "max_iter": 9910,
    "learning_rate": 0.0009347843785877883,
    "max_depth": 20,
    "min_samples_leaf": 218,
    "max_features": 0.31912233783575017,
    "max_leaf_nodes": 437,
    "l2_regularization": 55.83966409360982,
    "max_bins": 231,
    "random_state": 42,
    "verbose": 0,
}

# Dictionary of models with hyperparameter tuning for full dataset with preprocessors
models_tuned_full_cat = {
    "XGBoost": (passthrough_preprocessor, XGBClassifier(**xgboost_full_cat_params)),
    "LightGBM": (passthrough_preprocessor, LGBMClassifier(**lightgbm_full_cat_params)),
    "CatBoost": (
        catboost_preprocessor,
        CatBoostClassifier(**catboost_full_cat_params),
    ),
    "HistGBM": (
        numeric_preprocessor_cat,
        HistGradientBoostingClassifier(**histgbm_full_cat_params),
    ),
    "Dummy - Most Frequent": (numeric_preprocessor_cat, DummyClassifier(strategy="prior"))
}

### Evaluation function

In [12]:
import time
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight
import warnings

# Suppress warnings caused by unseen categories in the test set.
warnings.filterwarnings(
    "ignore",
    message="Found unknown categories in columns.*",
    category=UserWarning,
    module="sklearn.preprocessing._encoders",
)

# Suppress scikit-learn pipeline compatibility warning.
warnings.filterwarnings(
    "ignore",
    message="This Pipeline instance is not fitted yet.*",
    category=FutureWarning,
    module="sklearn.pipeline",
)


# Function to evaluate models on test and train sets, compute metrics, and store predictions
def compare_models_cat(
    models_dict, train_X, train_y, test_X, test_y, dataset_size_name, tuning_status
):
    # Prepare output directories
    out_dir_scores = Path("test_results/Classification/scores")
    out_dir_preds = Path("test_results/Classification/predictions")

    # Create output variables
    results_scores_test = []
    results_scores_train = []

    test_preds_df = pd.DataFrame(index=test_X.index)
    test_preds_df["true_y"] = test_y
    train_preds_df = pd.DataFrame(index=train_X.index)
    train_preds_df["true_y"] = train_y

    # Evaluate each model
    for model_name, (preprocessor, model) in models_dict.items():
        print(
            f"Evaluating {model_name} on {dataset_size_name} dataset ({tuning_status})..."
        )
        pipe = Pipeline([("preprocessor", preprocessor), ("classifier", model)])

        # Fit the model
        start_time_fit = time.time()
        # For NGBoost, compute sample weights to handle class imbalance
        if "NGBoost" in model_name:
            weights = compute_sample_weight(class_weight="balanced", y=train_y)
            pipe.fit(train_X, train_y, classifier__sample_weight=weights)
        else:
            pipe.fit(train_X, train_y)
        fit_time = time.time() - start_time_fit

        # Get number of trees to compute time per tree
        actual_model = pipe.named_steps["classifier"]
        n_trees = get_number_of_trees(model_name, actual_model)
        n_trees = n_trees if n_trees is not None else 1

        # Predict probabilities and compute metrics for test set
        start_time_pred_test = time.time()
        test_pred_proba = pipe.predict_proba(test_X)[:, 1]
        test_pred_time = time.time() - start_time_pred_test
        test_roc_auc = roc_auc_score(test_y, test_pred_proba)
        test_pr_auc = average_precision_score(test_y, test_pred_proba)

        # Predict probabilities and compute metrics for train set
        start_time_pred_train = time.time()
        train_pred_proba = pipe.predict_proba(train_X)[:, 1]
        train_pred_time = time.time() - start_time_pred_train
        train_roc_auc = roc_auc_score(train_y, train_pred_proba)
        train_pr_auc = average_precision_score(train_y, train_pred_proba)

        # Save predictions and results
        test_preds_df[model_name] = test_pred_proba
        train_preds_df[model_name] = train_pred_proba
        fit_time_per_tree = fit_time / n_trees * 1000
        test_pred_time_per_tree = test_pred_time / n_trees * 1000
        train_pred_time_per_tree = train_pred_time / n_trees * 1000

        results_scores_test.append(
            {
                "Dataset": dataset_size_name,
                "Model": model_name,
                "ROC AUC": test_roc_auc,
                "PR AUC": test_pr_auc,
                "Fit Time (s)": fit_time,
                "Pred Time (s)": test_pred_time,
                "Fit Time (ms/tree)": fit_time_per_tree,
                "Pred Time (ms/tree)": test_pred_time_per_tree,
                "Trees": n_trees,
            }
        )
        results_scores_train.append(
            {
                "Dataset": dataset_size_name,
                "Model": model_name,
                "ROC AUC": train_roc_auc,
                "PR AUC": train_pr_auc,
                "Fit Time (s)": fit_time,
                "Pred Time (s)": train_pred_time,
                "Fit Time (ms/tree)": fit_time_per_tree,
                "Pred Time (ms/tree)": train_pred_time_per_tree,
                "Trees": n_trees,
            }
        )

    # Convert results to DataFrames
    df_scores_test = pd.DataFrame(results_scores_test)
    df_scores_train = pd.DataFrame(results_scores_train)

    # Save predictions and scores to files
    test_preds_path = (
        out_dir_preds / f"test_preds_{dataset_size_name}_{tuning_status}.parquet"
    )
    train_preds_path = (
        out_dir_preds / f"train_preds_{dataset_size_name}_{tuning_status}.parquet"
    )
    test_scores_path = (
        out_dir_scores / f"test_scores_{dataset_size_name}_{tuning_status}.csv"
    )
    train_scores_path = (
        out_dir_scores / f"train_scores_{dataset_size_name}_{tuning_status}.csv"
    )

    test_preds_df.to_parquet(test_preds_path)
    train_preds_df.to_parquet(train_preds_path)
    df_scores_test.to_csv(test_scores_path, index=False)
    df_scores_train.to_csv(train_scores_path, index=False)

    # Show results
    print(
        f"Finished evaluating {dataset_size_name} dataset ({tuning_status}). Results saved to {out_dir_scores}/ and {out_dir_preds}/."
    )
    display(df_scores_train)
    display(df_scores_test)

### Testing

In [12]:
compare_models_cat(models_no_tuning_cat, train_1k_X, train_1k_y_cat, test_full_X, test_full_y_cat, "1k", "not_tuned")

Evaluating XGBoost on 1k dataset (not_tuned)...
Evaluating LightGBM on 1k dataset (not_tuned)...
Evaluating CatBoost on 1k dataset (not_tuned)...
Evaluating NGBoost on 1k dataset (not_tuned)...
Evaluating GBM on 1k dataset (not_tuned)...
Evaluating HistGBM on 1k dataset (not_tuned)...
Evaluating Dummy - Most Frequent on 1k dataset (not_tuned)...
Finished evaluating 1k dataset (not_tuned). Results saved to test_results/Classification/scores/ and test_results/Classification/predictions/.


,Dataset,Model,ROC AUC,PR AUC,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,1k,XGBoost,1.000000,1.000000,0.147033,0.002904,1.470332,0.029039,100
1,1k,LightGBM,1.000000,1.000000,0.394042,0.004139,3.940418,0.041389,100
2,1k,CatBoost,0.998963,0.996943,2.381933,0.005324,2.381933,0.005324,1000
3,1k,NGBoost,0.910111,0.775710,4.331288,0.068157,8.662576,0.136314,500
4,1k,GBM,0.982595,0.961621,0.738164,0.007926,7.381639,0.079260,100
5,1k,HistGBM,1.000000,1.000000,0.200972,0.006540,2.009721,0.065401,100
6,1k,Dummy - Most Frequent,0.500000,0.227000,0.005581,0.003283,5.581141,3.283024,1


,Dataset,Model,ROC AUC,PR AUC,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,1k,XGBoost,0.650179,0.322478,0.147033,0.127206,1.470332,1.272061,100
1,1k,LightGBM,0.656751,0.325501,0.394042,0.444490,3.940418,4.444900,100
2,1k,CatBoost,0.678580,0.347285,2.381933,0.373577,2.381933,0.373577,1000
3,1k,NGBoost,0.674353,0.333852,4.331288,10.610609,8.662576,21.221218,500
4,1k,GBM,0.663176,0.326703,0.738164,1.265150,7.381639,12.651503,100
5,1k,HistGBM,0.659843,0.326385,0.200972,0.812530,2.009721,8.125300,100
6,1k,Dummy - Most Frequent,0.500000,0.214253,0.005581,0.293428,5.581141,293.427944,1


In [13]:
compare_models_cat(models_no_tuning_cat, train_10k_X, train_10k_y_cat, test_full_X, test_full_y_cat, "10k", "not_tuned")

Evaluating XGBoost on 10k dataset (not_tuned)...
Evaluating LightGBM on 10k dataset (not_tuned)...
Evaluating CatBoost on 10k dataset (not_tuned)...
Evaluating NGBoost on 10k dataset (not_tuned)...
Evaluating GBM on 10k dataset (not_tuned)...
Evaluating HistGBM on 10k dataset (not_tuned)...
Evaluating Dummy - Most Frequent on 10k dataset (not_tuned)...
Finished evaluating 10k dataset (not_tuned). Results saved to test_results/Classification/scores/ and test_results/Classification/predictions/.


,Dataset,Model,ROC AUC,PR AUC,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,10k,XGBoost,0.999812,0.999457,0.281938,0.008249,2.819378,0.082490,100
1,10k,LightGBM,0.979322,0.946979,0.481243,0.012950,4.812429,0.129499,100
2,10k,CatBoost,0.929348,0.853889,6.546292,0.017707,6.546292,0.017707,1000
3,10k,NGBoost,0.756382,0.480243,39.553584,0.340217,79.107168,0.680434,500
4,10k,GBM,0.778505,0.546173,6.912071,0.042226,69.120712,0.422258,100
5,10k,HistGBM,0.969711,0.928897,0.332293,0.021871,3.322930,0.218713,100
6,10k,Dummy - Most Frequent,0.500000,0.220400,0.014983,0.010183,14.983177,10.183096,1


,Dataset,Model,ROC AUC,PR AUC,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,10k,XGBoost,0.665033,0.336802,0.281938,0.152378,2.819378,1.523781,100
1,10k,LightGBM,0.696342,0.368995,0.481243,0.261684,4.812429,2.616839,100
2,10k,CatBoost,0.710763,0.386323,6.546292,0.336046,6.546292,0.336046,1000
3,10k,NGBoost,0.706233,0.384054,39.553584,10.622926,79.107168,21.245852,500
4,10k,GBM,0.705737,0.380528,6.912071,1.283153,69.120712,12.831531,100
5,10k,HistGBM,0.696229,0.366994,0.332293,0.597256,3.322930,5.972559,100
6,10k,Dummy - Most Frequent,0.500000,0.214253,0.014983,0.295453,14.983177,295.453072,1


In [14]:
compare_models_cat(models_no_tuning_cat, train_100k_X, train_100k_y_cat, test_full_X, test_full_y_cat, "100k", "not_tuned")

Evaluating XGBoost on 100k dataset (not_tuned)...
Evaluating LightGBM on 100k dataset (not_tuned)...
Evaluating CatBoost on 100k dataset (not_tuned)...
Evaluating NGBoost on 100k dataset (not_tuned)...
Evaluating GBM on 100k dataset (not_tuned)...
Evaluating HistGBM on 100k dataset (not_tuned)...
Evaluating Dummy - Most Frequent on 100k dataset (not_tuned)...
Finished evaluating 100k dataset (not_tuned). Results saved to test_results/Classification/scores/ and test_results/Classification/predictions/.


,Dataset,Model,ROC AUC,PR AUC,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,100k,XGBoost,0.878127,0.739829,1.311725,0.062981,13.117249,0.629811,100
1,100k,LightGBM,0.780007,0.555470,1.018264,0.124179,10.182641,1.241789,100
2,100k,CatBoost,0.813665,0.635952,29.138535,0.130899,29.138535,0.130899,1000
3,100k,NGBoost,0.717233,0.456858,402.245773,4.031801,804.491546,8.063602,500
4,100k,GBM,0.723350,0.465863,69.179656,0.442504,691.796563,4.425039,100
5,100k,HistGBM,0.755271,0.517057,1.438953,0.177509,14.389529,1.775091,100
6,100k,Dummy - Most Frequent,0.500000,0.248090,0.107763,0.075807,107.763052,75.807095,1


,Dataset,Model,ROC AUC,PR AUC,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,100k,XGBoost,0.706904,0.386137,1.311725,0.177451,13.117249,1.774511,100
1,100k,LightGBM,0.724336,0.408414,1.018264,0.278147,10.182641,2.781470,100
2,100k,CatBoost,0.729006,0.412856,29.138535,0.332134,29.138535,0.332134,1000
3,100k,NGBoost,0.716817,0.398827,402.245773,10.865097,804.491546,21.730194,500
4,100k,GBM,0.719301,0.402044,69.179656,1.239014,691.796563,12.390139,100
5,100k,HistGBM,0.723843,0.407878,1.438953,0.506740,14.389529,5.067399,100
6,100k,Dummy - Most Frequent,0.500000,0.214253,0.107763,0.238852,107.763052,238.852262,1


In [15]:
compare_models_cat(models_no_tuning_cat, train_full_X, train_full_y_cat, test_full_X, test_full_y_cat, "full", "not_tuned")

Evaluating XGBoost on full dataset (not_tuned)...
Evaluating LightGBM on full dataset (not_tuned)...
Evaluating CatBoost on full dataset (not_tuned)...
Evaluating NGBoost on full dataset (not_tuned)...
Evaluating GBM on full dataset (not_tuned)...
Evaluating HistGBM on full dataset (not_tuned)...
Evaluating Dummy - Most Frequent on full dataset (not_tuned)...
Finished evaluating full dataset (not_tuned). Results saved to test_results/Classification/scores/ and test_results/Classification/predictions/.


,Dataset,Model,ROC AUC,PR AUC,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,full,XGBoost,0.768326,0.464807,10.408532,0.682891,104.085319,6.828909,100
1,full,LightGBM,0.742379,0.418292,5.810439,1.288761,58.104389,12.887611,100
2,full,CatBoost,0.767558,0.476952,283.296990,1.489083,283.296990,1.489083,1000
3,full,NGBoost,0.725828,0.391655,4529.797635,43.251696,9059.595270,86.503392,500
4,full,GBM,0.728389,0.396295,711.736131,6.479409,7117.361312,64.794092,100
5,full,HistGBM,0.740450,0.414633,22.496724,2.428951,224.967239,24.289510,100
6,full,Dummy - Most Frequent,0.500000,0.194276,1.098659,0.802590,1098.659277,802.589893,1


,Dataset,Model,ROC AUC,PR AUC,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,full,XGBoost,0.721658,0.403980,10.408532,0.163582,104.085319,1.635821,100
1,full,LightGBM,0.721926,0.405490,5.810439,0.381646,58.104389,3.816459,100
2,full,CatBoost,0.725213,0.408408,283.296990,0.386749,283.296990,0.386749,1000
3,full,NGBoost,0.706670,0.382903,4529.797635,11.102384,9059.595270,22.204768,500
4,full,GBM,0.709250,0.388326,711.736131,1.235903,7117.361312,12.359030,100
5,full,HistGBM,0.721001,0.404607,22.496724,0.723194,224.967239,7.231941,100
6,full,Dummy - Most Frequent,0.500000,0.214253,1.098659,0.238186,1098.659277,238.186121,1


In [16]:
compare_models_cat(models_tuned_1k_cat, train_1k_X, train_1k_y_cat, test_full_X, test_full_y_cat, "1k", "tuned")

Evaluating XGBoost on 1k dataset (tuned)...
Evaluating LightGBM on 1k dataset (tuned)...
Evaluating CatBoost on 1k dataset (tuned)...
Evaluating NGBoost on 1k dataset (tuned)...
Evaluating GBM on 1k dataset (tuned)...
Evaluating HistGBM on 1k dataset (tuned)...
Evaluating Dummy - Most Frequent on 1k dataset (tuned)...
Finished evaluating 1k dataset (tuned). Results saved to test_results/Classification/scores/ and test_results/Classification/predictions/.


,Dataset,Model,ROC AUC,PR AUC,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,1k,XGBoost,0.996854,0.987822,0.392872,0.003640,0.639857,0.005929,614
1,1k,LightGBM,0.923486,0.789284,0.198002,0.006114,0.532263,0.016436,372
2,1k,CatBoost,0.961538,0.889620,0.841240,0.004884,1.942817,0.011279,433
3,1k,NGBoost,0.957720,0.878772,1.279511,0.083933,1.736107,0.113885,737
4,1k,GBM,0.820876,0.591594,0.246648,0.007584,0.425990,0.013099,579
5,1k,HistGBM,0.972372,0.921584,0.429077,0.014445,0.530380,0.017855,809
6,1k,Dummy - Most Frequent,0.500000,0.227000,0.005315,0.003297,5.315065,3.297091,1


,Dataset,Model,ROC AUC,PR AUC,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,1k,XGBoost,0.668767,0.337649,0.392872,0.322539,0.639857,0.525308,614
1,1k,LightGBM,0.683716,0.352338,0.198002,1.011673,0.532263,2.719551,372
2,1k,CatBoost,0.679436,0.349067,0.841240,0.310369,1.942817,0.716787,433
3,1k,NGBoost,0.680578,0.349238,1.279511,9.645949,1.736107,13.088126,737
4,1k,GBM,0.685286,0.355600,0.246648,1.166788,0.425990,2.015178,579
5,1k,HistGBM,0.676686,0.345461,0.429077,2.055733,0.530380,2.541079,809
6,1k,Dummy - Most Frequent,0.500000,0.214253,0.005315,0.294004,5.315065,294.004202,1


In [17]:
compare_models_cat(models_tuned_10k_cat, train_10k_X, train_10k_y_cat, test_full_X, test_full_y_cat, "10k", "tuned")

Evaluating XGBoost on 10k dataset (tuned)...
Evaluating LightGBM on 10k dataset (tuned)...
Evaluating CatBoost on 10k dataset (tuned)...
Evaluating NGBoost on 10k dataset (tuned)...
Evaluating GBM on 10k dataset (tuned)...
Evaluating HistGBM on 10k dataset (tuned)...
Evaluating Dummy - Most Frequent on 10k dataset (tuned)...
Finished evaluating 10k dataset (tuned). Results saved to test_results/Classification/scores/ and test_results/Classification/predictions/.


,Dataset,Model,ROC AUC,PR AUC,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,10k,XGBoost,0.742388,0.455275,2.163722,0.035171,1.183655,0.019240,1828
1,10k,LightGBM,0.808811,0.548836,1.771554,0.187744,1.733419,0.183703,1022
2,10k,CatBoost,0.731490,0.435792,7.426045,0.040467,7.396459,0.040306,1004
3,10k,NGBoost,0.752344,0.467446,8.882577,0.484229,12.134668,0.661515,732
4,10k,GBM,0.744714,0.461295,6.029846,0.186214,3.797132,0.117263,1588
5,10k,HistGBM,0.750469,0.472544,2.116023,0.081046,1.565106,0.059945,1352
6,10k,Dummy - Most Frequent,0.500000,0.220400,0.014877,0.009864,14.877081,9.863853,1


,Dataset,Model,ROC AUC,PR AUC,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,10k,XGBoost,0.711726,0.391639,2.163722,0.871667,1.183655,0.476842,1828
1,10k,LightGBM,0.713658,0.392560,1.771554,4.928084,1.733419,4.822000,1022
2,10k,CatBoost,0.706193,0.382960,7.426045,0.811621,7.396459,0.808387,1004
3,10k,NGBoost,0.711784,0.391869,8.882577,13.818173,12.134668,18.877286,732
4,10k,GBM,0.710947,0.389370,6.029846,6.591866,3.797132,4.151049,1588
5,10k,HistGBM,0.710174,0.388450,2.116023,2.066853,1.565106,1.528737,1352
6,10k,Dummy - Most Frequent,0.500000,0.214253,0.014877,0.293705,14.877081,293.704987,1


In [14]:
compare_models_cat(models_tuned_100k_cat, train_100k_X, train_100k_y_cat, test_full_X, test_full_y_cat, "100k", "tuned")

Evaluating XGBoost on 100k dataset (tuned)...
Evaluating LightGBM on 100k dataset (tuned)...
Evaluating CatBoost on 100k dataset (tuned)...
Evaluating NGBoost on 100k dataset (tuned)...
Evaluating GBM on 100k dataset (tuned)...
Evaluating HistGBM on 100k dataset (tuned)...
Evaluating Dummy - Most Frequent on 100k dataset (tuned)...
Finished evaluating 100k dataset (tuned). Results saved to test_results/Classification/scores/ and test_results/Classification/predictions/.


,Dataset,Model,ROC AUC,PR AUC,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,100k,XGBoost,0.780962,0.542409,135.167573,12.638306,15.001950,1.402698,9010
1,100k,LightGBM,0.806156,0.582896,79.111436,42.767469,9.089090,4.913542,8704
2,100k,CatBoost,0.765740,0.514220,1072.544998,0.761463,119.971476,0.085175,8940
3,100k,NGBoost,0.784230,0.552549,1735.955274,73.541471,190.764316,8.081480,9100
4,100k,GBM,0.725892,0.468303,3945.958165,2.099629,830.728035,0.442027,4750
5,100k,HistGBM,0.794320,0.569805,49.345217,6.259513,12.556035,1.592751,3930
6,100k,Dummy - Most Frequent,0.500000,0.248090,0.106808,0.075246,106.808186,75.246096,1


,Dataset,Model,ROC AUC,PR AUC,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,100k,XGBoost,0.730452,0.416618,135.167573,33.683487,15.001950,3.738456,9010
1,100k,LightGBM,0.731406,0.418288,79.111436,112.083696,9.089090,12.877263,8704
2,100k,CatBoost,0.729827,0.416037,1072.544998,2.399769,119.971476,0.268431,8940
3,100k,NGBoost,0.730501,0.417289,1735.955274,195.351951,190.764316,21.467247,9100
4,100k,GBM,0.722385,0.406817,3945.958165,6.058117,830.728035,1.275393,4750
5,100k,HistGBM,0.728802,0.414570,49.345217,17.125149,12.556035,4.357544,3930
6,100k,Dummy - Most Frequent,0.500000,0.214253,0.106808,0.240904,106.808186,240.904331,1


In [15]:
compare_models_cat(models_tuned_full_cat, train_full_X, train_full_y_cat, test_full_X, test_full_y_cat, "full", "tuned")

Evaluating XGBoost on full dataset (tuned)...
Evaluating LightGBM on full dataset (tuned)...
Evaluating CatBoost on full dataset (tuned)...
Evaluating HistGBM on full dataset (tuned)...
Evaluating Dummy - Most Frequent on full dataset (tuned)...
Finished evaluating full dataset (tuned). Results saved to test_results/Classification/scores/ and test_results/Classification/predictions/.


,Dataset,Model,ROC AUC,PR AUC,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,full,XGBoost,0.764362,0.453091,729.670938,81.075507,75.457181,8.384230,9670
1,full,LightGBM,0.775721,0.453930,1031.231801,928.796116,106.642379,96.049236,9670
2,full,CatBoost,0.747242,0.420575,5686.941159,6.284548,659.737953,0.729066,8620
3,full,HistGBM,0.784421,0.489819,5213.599842,410.312648,526.094838,41.403900,9910
4,full,Dummy - Most Frequent,0.500000,0.194276,1.109566,0.792906,1109.565973,792.905807,1


,Dataset,Model,ROC AUC,PR AUC,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,full,XGBoost,0.730010,0.416948,729.670938,18.025538,75.457181,1.864068,9670
1,full,LightGBM,0.728599,0.413425,1031.231801,226.941852,106.642379,23.468651,9670
2,full,CatBoost,0.726016,0.410475,5686.941159,1.631784,659.737953,0.189302,8620
3,full,HistGBM,0.727049,0.412820,5213.599842,102.730662,526.094838,10.366363,9910
4,full,Dummy - Most Frequent,0.500000,0.214253,1.109566,0.260547,1109.565973,260.546923,1


## Regression

In [12]:
# Changes in preprossing for regression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.pipeline import make_pipeline

# Print categorical columns
cols_to_nominal_cat = train_df.select_dtypes(
    include=["object", "category"]
).columns.tolist()
print("Categorical columns:")
for col in cols_to_nominal_cat:
    print(f"- {col}")

# Set maximal cardinality for one-hot encoding
cardinality = train_df[cols_to_nominal_cat].nunique()
threshold_for_ohe = 5
cols_for_ohe = cardinality[cardinality <= threshold_for_ohe].index.tolist()
cols_for_te = cardinality[cardinality > threshold_for_ohe].index.tolist()
ohe_categories = []
for col in cols_for_ohe:
    unique_cats = train_df[col].dropna().unique().tolist()
    ohe_categories.append(unique_cats)

# Create OneHotEncoder
ohe_transformer = OneHotEncoder(
    categories=ohe_categories,
    drop="if_binary",
    handle_unknown="ignore",
    sparse_output=False,
)

# Create TargetEncoder for rest of nominal categorical features
target_transformer_nominal = TargetEncoder(target_type="continuous", smooth="auto")

# Create whole preprocessor for models which are unable to handle categorical
# features but able to handle missing values
numeric_preprocessor_reg = ColumnTransformer(
    transformers=[
        ("ohe", ohe_transformer, cols_for_ohe),
        ("target_enc", target_transformer_nominal, cols_for_te),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

# Create whole preprocessor for GBM with imputation based on hyperparameter tuning
imputed_numeric_preprocessor_1k_reg = make_pipeline(
    numeric_preprocessor_reg, SimpleImputer(strategy="mean")
).set_output(transform="pandas")

imputed_numeric_preprocessor_10k_reg = make_pipeline(
    numeric_preprocessor_reg, IterativeImputer(max_iter=10, random_state=42)
).set_output(transform="pandas")

imputed_numeric_preprocessor_100k_reg = make_pipeline(
    numeric_preprocessor_reg, IterativeImputer(max_iter=10, random_state=42)
).set_output(transform="pandas")

# Create passthrough preprocessor for models which are able to handle
# categorical features and missing values
passthrough_preprocessor = "passthrough"

# Create preprocessor for CatBoost filling missing values of categorical
# features with "Missing"
catboost_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat_imputer",
            SimpleImputer(strategy="constant", fill_value="Missing"),
            cols_to_nominal_cat,
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

Categorical columns:
- home_ownership
- verification_status
- purpose
- addr_state
- initial_list_status
- application_type
- disbursement_method


### Model definition

In [13]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from ngboost import NGBRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor as HistGBMRegressor
from pgbm.sklearn import HistGradientBoostingRegressor as PGBMRegressor
from ngboost.distns import Normal
from sklearn.tree import DecisionTreeRegressor

models_no_tuning_reg = {
    "XGBoost": (
        passthrough_preprocessor,
        XGBRegressor(random_state=42, enable_categorical=True),
    ),
    "LightGBM": (passthrough_preprocessor, LGBMRegressor(random_state=42, verbose=-1)),
    "CatBoost": (
        catboost_preprocessor,
        CatBoostRegressor(random_state=42, verbose=0, cat_features=cols_to_nominal_cat),
    ),
    "NGBoost": (numeric_preprocessor_reg, NGBRegressor(random_state=42, verbose=False)),
    "GBM": (
        imputed_numeric_preprocessor_1k_reg,
        GradientBoostingRegressor(random_state=42),
    ),
    "HistGBM": (numeric_preprocessor_reg, HistGBMRegressor(random_state=42)),
    "PGBM": (numeric_preprocessor_reg, PGBMRegressor(random_state=42)),
    "Dummy - Mean": (numeric_preprocessor_reg, DummyRegressor(strategy="mean")),
}

In [14]:
xgboost_1k_reg_params = {
    "booster": "gbtree",
    "learning_rate": 0.006592924714393806,
    "min_split_loss": 0.3399903461254572,
    "max_depth": 7,
    "min_child_weight": 3.6857527823293634,
    "max_delta_step": 3.928223239451256,
    "colsample_bytree": 0.381061252694662,
    "colsample_bylevel": 0.3290330497906662,
    "colsample_bynode": 0.3640630952340392,
    "reg_lambda": 0.01062303202535693,
    "reg_alpha": 0.03812077743370024,
    "grow_policy": "lossguide",
    "max_leaves": 20,
    "max_bin": 169,
    "max_cat_to_onehot": 18,
    "max_cat_threshold": 11,
    "n_estimators": 537,
    "sampling_method": "uniform",
    "subsample": 0.5636865373558122,
    "random_state": 42,
    "n_jobs": n_cores,
    "verbosity": 0,
    "tree_method": "hist",
    "objective": "reg:squarederror",
    "enable_categorical": True,
}
lightgbm_1k_reg_params = {
    "boosting_type": "gbdt",
    "num_leaves": 13,
    "max_depth": 3,
    "learning_rate": 0.02169809132057075,
    "min_split_gain": 0.027389027279699385,
    "min_child_weight": 0.6899288118460188,
    "min_child_samples": 42,
    "colsample_bytree": 0.37238284939773186,
    "reg_alpha": 0.09371206330869274,
    "reg_lambda": 0.019391247189888383,
    "colsample_bynode": 0.30902821981225653,
    "min_data_per_group": 33,
    "max_cat_threshold": 29,
    "cat_l2": 0.041250555077113255,
    "cat_smooth": 28.95327386652521,
    "max_cat_to_onehot": 6,
    "max_bin": 161,
    "n_estimators": 144,
    "subsample": 0.8693945506758759,
    "subsample_freq": 5,
    "random_state": 42,
    "n_jobs": n_cores,
    "verbose": -1,
    "objective": "regression",
    "metric": "rmse",
}
catboost_1k_reg_params = {
    "iterations": 976,
    "learning_rate": 0.00976818825318037,
    "depth": 4,
    "l2_leaf_reg": 0.027265370477102587,
    "random_strength": 0.7031697546586838,
    "rsm": 0.8385396695909164,
    "border_count": 201,
    "boosting_type": "Ordered",
    "bootstrap_type": "Bayesian",
    "score_function": "Cosine",
    "bagging_temperature": 9.530476491737797,
    "random_state": 42,
    "thread_count": n_cores,
    "verbose": False,
    "objective": "RMSE",
    "cat_features": [
        "home_ownership",
        "verification_status",
        "purpose",
        "addr_state",
        "initial_list_status",
        "application_type",
        "disbursement_method",
    ],
}
ngboost_1k_reg_params = {
    "n_estimators": 263,
    "learning_rate": 0.015377425834686074,
    "minibatch_frac": 0.663571492569765,
    "col_sample": 0.5226236509136446,
    "random_state": 42,
    "verbose": False,
}
gbm_1k_reg_params = {
    "n_estimators": 971,
    "learning_rate": 0.01388668367558402,
    "max_depth": 8,
    "subsample": 0.30872672854426325,
    "min_samples_split": 43,
    "min_samples_leaf": 28,
    "min_weight_fraction_leaf": 0.3945207700562416,
    "min_impurity_decrease": 0.16016521565758096,
    "max_features": "sqrt",
    "max_leaf_nodes": 6,
    "ccp_alpha": 3.910485014559049e-05,
    "random_state": 42,
    "verbose": 0,
}
histgbm_1k_reg_params = {
    "max_iter": 734,
    "learning_rate": 0.006736563678671684,
    "max_depth": 2,
    "min_samples_leaf": 52,
    "max_features": 0.203174979863919,
    "max_leaf_nodes": 6,
    "l2_regularization": 0.0063066439175882,
    "max_bins": 124,
    "random_state": 42,
    "verbose": 0,
}
pgbm_1k_reg_params = {
    "max_iter": 266,
    "learning_rate": 0.0057633608637110215,
    "max_depth": 2,
    "min_samples_leaf": 80,
    "max_leaf_nodes": 24,
    "l2_regularization": 3.757726025937608e-05,
    "max_bins": 106,
    "tree_correlation": 0.00017249809953907554,
    "distribution": "laplace",
    "random_state": 42,
    "verbose": 0,
    "with_variance": True,
}
ngboost_1k_reg_tree_regressor_params = {
    "criterion": "friedman_mse",
    "max_depth": 8,
    "min_samples_split": 25,
    "min_samples_leaf": 29,
    "min_weight_fraction_leaf": 0.10108332219836896,
    "max_features": "log2",
    "ccp_alpha": 0.00011946599313498593,
    "max_leaf_nodes": 6,
    "min_impurity_decrease": 0.289818830272591,
    "random_state": 42,
}
ngboost_1k_reg_tree_regressor = DecisionTreeRegressor(
    **ngboost_1k_reg_tree_regressor_params
)

models_tuned_1k_reg = {
    "XGBoost": (passthrough_preprocessor, XGBRegressor(**xgboost_1k_reg_params)),
    "LightGBM": (passthrough_preprocessor, LGBMRegressor(**lightgbm_1k_reg_params)),
    "CatBoost": (catboost_preprocessor, CatBoostRegressor(**catboost_1k_reg_params)),
    "NGBoost": (
        numeric_preprocessor_reg,
        NGBRegressor(
            **ngboost_1k_reg_params, Dist=Normal, Base=ngboost_1k_reg_tree_regressor
        ),
    ),
    "GBM": (
        imputed_numeric_preprocessor_1k_reg,
        GradientBoostingRegressor(**gbm_1k_reg_params),
    ),
    "HistGBM": (numeric_preprocessor_reg, HistGBMRegressor(**histgbm_1k_reg_params)),
    "PGBM": (numeric_preprocessor_reg, PGBMRegressor(**pgbm_1k_reg_params)),
    "Dummy - Mean": (numeric_preprocessor_reg, DummyRegressor(strategy="mean")),
}

In [15]:
xgboost_10k_reg_params = {
    "booster": "gbtree",
    "learning_rate": 0.0034649471654673372,
    "min_split_loss": 1.2225056392066438,
    "max_depth": 4,
    "min_child_weight": 0.044868703229338794,
    "max_delta_step": 7.322896593525407,
    "colsample_bytree": 0.664513006915934,
    "colsample_bylevel": 0.30183598693553104,
    "colsample_bynode": 0.6199112663715858,
    "reg_lambda": 0.022303464542099465,
    "reg_alpha": 8.45346616493749,
    "grow_policy": "depthwise",
    "max_leaves": 31,
    "max_bin": 69,
    "max_cat_to_onehot": 3,
    "max_cat_threshold": 32,
    "n_estimators": 1990,
    "sampling_method": "gradient_based",
    "subsample": 0.19636351824285336,
    "random_state": 42,
    "n_jobs": n_cores,
    "verbosity": 0,
    "tree_method": "hist",
    "objective": "reg:squarederror",
    "enable_categorical": True,
}
lightgbm_10k_reg_params = {
    "boosting_type": "goss",
    "num_leaves": 28,
    "max_depth": 3,
    "learning_rate": 0.00573759926773727,
    "min_split_gain": 0.041494825179870576,
    "min_child_weight": 0.012349230729538438,
    "min_child_samples": 81,
    "colsample_bytree": 0.5283693462952471,
    "reg_alpha": 9.158641225780748,
    "reg_lambda": 0.004093641172128938,
    "colsample_bynode": 0.6725807827106883,
    "min_data_per_group": 91,
    "max_cat_threshold": 6,
    "cat_l2": 1.5908743669801766,
    "cat_smooth": 0.14729665520231705,
    "max_cat_to_onehot": 11,
    "max_bin": 156,
    "n_estimators": 1494,
    "top_rate": 0.1092716955430614,
    "other_rate": 0.777449860751789,
    "random_state": 42,
    "n_jobs": n_cores,
    "verbose": -1,
    "objective": "regression",
    "metric": "rmse",
}
catboost_10k_reg_params = {
    "iterations": 1088,
    "learning_rate": 0.013825345653490797,
    "depth": 5,
    "l2_leaf_reg": 0.5148645656737443,
    "random_strength": 0.1893828349911231,
    "rsm": 0.7999433598790255,
    "border_count": 194,
    "boosting_type": "Ordered",
    "bootstrap_type": "Bayesian",
    "score_function": "L2",
    "bagging_temperature": 8.170909952414544,
    "random_state": 42,
    "thread_count": n_cores,
    "verbose": False,
    "objective": "RMSE",
    "cat_features": [
        "home_ownership",
        "verification_status",
        "purpose",
        "addr_state",
        "initial_list_status",
        "application_type",
        "disbursement_method",
    ],
}
ngboost_10k_reg_params = {
    "n_estimators": 1428,
    "learning_rate": 0.005743155031743333,
    "minibatch_frac": 0.5442236810583654,
    "col_sample": 0.7169685029737407,
    "random_state": 42,
    "verbose": False,
}
gbm_10k_reg_params = {
    "n_estimators": 1876,
    "learning_rate": 0.022112077171996453,
    "max_depth": 3,
    "subsample": 0.4440702904540561,
    "min_samples_split": 16,
    "min_samples_leaf": 40,
    "min_weight_fraction_leaf": 0.07834199132955552,
    "min_impurity_decrease": 0.7616676770612814,
    "max_features": "log2",
    "max_leaf_nodes": 22,
    "ccp_alpha": 1.5744809085067783e-05,
    "random_state": 42,
    "verbose": 0,
}
histgbm_10k_reg_params = {
    "max_iter": 878,
    "learning_rate": 0.004906344452687288,
    "max_depth": 5,
    "min_samples_leaf": 23,
    "max_features": 0.3270420473977633,
    "max_leaf_nodes": 5,
    "l2_regularization": 0.0003377040719829344,
    "max_bins": 71,
    "random_state": 42,
    "verbose": 0,
}
pgbm_10k_reg_params = {
    "max_iter": 670,
    "learning_rate": 0.0046421620560510345,
    "max_depth": 6,
    "min_samples_leaf": 26,
    "max_leaf_nodes": 8,
    "l2_regularization": 86.43849217392876,
    "max_bins": 149,
    "tree_correlation": 0.030672618536641574,
    "distribution": "laplace",
    "random_state": 42,
    "verbose": 0,
    "with_variance": True,
}
ngboost_10k_reg_tree_regressor_params = {
    "criterion": "friedman_mse",
    "max_depth": 8,
    "min_samples_split": 38,
    "min_samples_leaf": 41,
    "min_weight_fraction_leaf": 0.0003213921303004477,
    "max_features": "log2",
    "ccp_alpha": 1.8549133165544335e-05,
    "max_leaf_nodes": 17,
    "min_impurity_decrease": 0.38252906628451194,
    "random_state": 42,
}
ngboost_10k_reg_tree_regressor = DecisionTreeRegressor(
    **ngboost_10k_reg_tree_regressor_params
)

models_tuned_10k_reg = {
    "XGBoost": (passthrough_preprocessor, XGBRegressor(**xgboost_10k_reg_params)),
    "LightGBM": (passthrough_preprocessor, LGBMRegressor(**lightgbm_10k_reg_params)),
    "CatBoost": (catboost_preprocessor, CatBoostRegressor(**catboost_10k_reg_params)),
    "NGBoost": (
        numeric_preprocessor_reg,
        NGBRegressor(
            **ngboost_10k_reg_params, Dist=Normal, Base=ngboost_10k_reg_tree_regressor
        ),
    ),
    "GBM": (
        imputed_numeric_preprocessor_10k_reg,
        GradientBoostingRegressor(**gbm_10k_reg_params),
    ),
    "HistGBM": (numeric_preprocessor_reg, HistGBMRegressor(**histgbm_10k_reg_params)),
    "PGBM": (numeric_preprocessor_reg, PGBMRegressor(**pgbm_10k_reg_params)),
    "Dummy - Mean": (numeric_preprocessor_reg, DummyRegressor(strategy="mean")),
}

In [16]:
xgboost_100k_reg_params = {
    "booster": "gbtree",
    "learning_rate": 0.00358874054592766,
    "min_split_loss": 0.30665873968801743,
    "max_depth": 9,
    "min_child_weight": 0.058208746224353215,
    "max_delta_step": 6.600824779885242,
    "colsample_bytree": 0.31452203665302875,
    "colsample_bylevel": 0.7845215630136946,
    "colsample_bynode": 0.45580656447386975,
    "reg_lambda": 1.4591079946865122,
    "reg_alpha": 7.300918108293281,
    "grow_policy": "depthwise",
    "max_leaves": 311,
    "max_bin": 270,
    "max_cat_to_onehot": 25,
    "max_cat_threshold": 578,
    "n_estimators": 9240,
    "sampling_method": "uniform",
    "subsample": 0.892493539712304,
    "random_state": 42,
    "n_jobs": n_cores,
    "verbosity": 0,
    "tree_method": "hist",
    "objective": "reg:squarederror",
    "enable_categorical": True,
}
lightgbm_100k_reg_params = {
    "boosting_type": "gbdt",
    "num_leaves": 321,
    "max_depth": 7,
    "learning_rate": 0.0014957375416499211,
    "min_split_gain": 0.15169683434030254,
    "min_child_weight": 0.0043443738122844215,
    "min_child_samples": 129,
    "colsample_bytree": 0.5703676576888828,
    "reg_alpha": 0.0009598013297060651,
    "reg_lambda": 1.512993041670286e-06,
    "colsample_bynode": 0.40283945018201495,
    "min_data_per_group": 572,
    "max_cat_threshold": 557,
    "cat_l2": 0.1829638573103202,
    "cat_smooth": 72.95247318237497,
    "max_cat_to_onehot": 51,
    "max_bin": 341,
    "n_estimators": 6740,
    "subsample": 0.998415655290472,
    "subsample_freq": 7,
    "random_state": 42,
    "n_jobs": n_cores,
    "verbose": -1,
    "objective": "regression",
    "metric": "rmse",
}
catboost_100k_reg_params = {
    "iterations": 9930,
    "learning_rate": 0.001087680526009645,
    "depth": 9,
    "l2_leaf_reg": 25.36922452973143,
    "random_strength": 1.1801448166458588,
    "rsm": 0.7742077921478168,
    "border_count": 217,
    "boosting_type": "Plain",
    "bootstrap_type": "MVS",
    "grow_policy": "Depthwise",
    "min_data_in_leaf": 299,
    "score_function": "L2",
    "subsample": 0.269182776220681,
    "random_state": 42,
    "thread_count": n_cores,
    "verbose": False,
    "objective": "RMSE",
    "cat_features": [
        "home_ownership",
        "verification_status",
        "purpose",
        "addr_state",
        "initial_list_status",
        "application_type",
        "disbursement_method",
    ],
}
ngboost_100k_reg_params = {
    "n_estimators": 3540,
    "learning_rate": 0.0031005975042584922,
    "minibatch_frac": 0.5337656312465495,
    "col_sample": 0.7957079748440611,
    "random_state": 42,
    "verbose": False,
}
gbm_100k_reg_params = {
    "n_estimators": 6020,
    "learning_rate": 0.004590009050693113,
    "max_depth": 4,
    "subsample": 0.3702702077447556,
    "min_samples_split": 343,
    "min_samples_leaf": 242,
    "min_weight_fraction_leaf": 0.020346710861258076,
    "min_impurity_decrease": 0.8240878364564256,
    "max_features": None,
    "max_leaf_nodes": 320,
    "ccp_alpha": 3.276614763080726e-05,
    "random_state": 42,
    "verbose": 0,
}
histgbm_100k_reg_params = {
    "max_iter": 4800,
    "learning_rate": 0.001430411566426403,
    "max_depth": 8,
    "min_samples_leaf": 259,
    "max_features": 0.4398281267806546,
    "max_leaf_nodes": 43,
    "l2_regularization": 0.00024044149018689098,
    "max_bins": 147,
    "random_state": 42,
    "verbose": 0,
}
pgbm_100k_reg_params = {
    "max_iter": 9490,
    "learning_rate": 0.0069166877386120885,
    "max_depth": 10,
    "min_samples_leaf": 327,
    "max_leaf_nodes": 24,
    "l2_regularization": 14.804147651872958,
    "max_bins": 169,
    "tree_correlation": 0.012392279453673792,
    "distribution": "laplace",
    "random_state": 42,
    "verbose": 0,
    "with_variance": True,
}
ngboost_100k_reg_tree_regressor_params = {
    "criterion": "friedman_mse",
    "max_depth": 17,
    "min_samples_split": 35,
    "min_samples_leaf": 371,
    "min_weight_fraction_leaf": 0.003373989905557241,
    "max_features": "sqrt",
    "ccp_alpha": 2.1975874211460698e-05,
    "max_leaf_nodes": 30,
    "min_impurity_decrease": 0.035785241864927805,
    "random_state": 42,
}
ngboost_100k_reg_tree_regressor = DecisionTreeRegressor(
    **ngboost_100k_reg_tree_regressor_params
)

models_tuned_100k_reg = {
    "XGBoost": (passthrough_preprocessor, XGBRegressor(**xgboost_100k_reg_params)),
    "LightGBM": (passthrough_preprocessor, LGBMRegressor(**lightgbm_100k_reg_params)),
    "CatBoost": (catboost_preprocessor, CatBoostRegressor(**catboost_100k_reg_params)),
    "NGBoost": (
        numeric_preprocessor_reg,
        NGBRegressor(
            **ngboost_100k_reg_params, Dist=Normal, Base=ngboost_100k_reg_tree_regressor
        ),
    ),
    "GBM": (
        imputed_numeric_preprocessor_100k_reg,
        GradientBoostingRegressor(**gbm_100k_reg_params),
    ),
    "HistGBM": (numeric_preprocessor_reg, HistGBMRegressor(**histgbm_100k_reg_params)),
    "PGBM": (numeric_preprocessor_reg, PGBMRegressor(**pgbm_100k_reg_params)),
    "Dummy - Mean": (numeric_preprocessor_reg, DummyRegressor(strategy="mean")),
}

In [17]:
xgboost_full_reg_params = {
    "booster": "gbtree",
    "learning_rate": 0.0015361738364973562,
    "min_split_loss": 0.019158435834981884,
    "max_depth": 12,
    "min_child_weight": 5.46291036872166,
    "max_delta_step": 4.8592833061852465,
    "colsample_bytree": 0.78840555707391,
    "colsample_bylevel": 0.3400841955942318,
    "colsample_bynode": 0.7797771602511868,
    "reg_lambda": 0.001536038127856472,
    "reg_alpha": 0.00444179665468952,
    "grow_policy": "lossguide",
    "max_leaves": 303,
    "max_bin": 352,
    "max_cat_to_onehot": 18,
    "max_cat_threshold": 668,
    "n_estimators": 6800,
    "sampling_method": "uniform",
    "subsample": 0.9347372849225181,
    "random_state": 42,
    "n_jobs": n_cores,
    "verbosity": 0,
    "tree_method": "hist",
    "objective": "reg:squarederror",
    "enable_categorical": True,
}
lightgbm_full_reg_params = {
    "boosting_type": "gbdt",
    "num_leaves": 367,
    "max_depth": 9,
    "learning_rate": 0.0037106925990830716,
    "min_split_gain": 0.028541720097881224,
    "min_child_weight": 3.114973690234082e-05,
    "min_child_samples": 449,
    "colsample_bytree": 0.9583470008702835,
    "reg_alpha": 4.042104970937467,
    "reg_lambda": 4.013246511788023e-08,
    "colsample_bynode": 0.544229805808371,
    "min_data_per_group": 745,
    "max_cat_threshold": 240,
    "cat_l2": 1.4174693320549176e-05,
    "cat_smooth": 0.006490614096342806,
    "max_cat_to_onehot": 4,
    "max_bin": 97,
    "n_estimators": 5940,
    "subsample": 0.9848817124796165,
    "subsample_freq": 7,
    "random_state": 42,
    "n_jobs": n_cores,
    "verbose": -1,
    "objective": "regression",
    "metric": "rmse",
}
catboost_full_reg_params = {
    "iterations": 8780,
    "learning_rate": 0.0019516220841303295,
    "depth": 9,
    "l2_leaf_reg": 2.9869534553308876e-07,
    "random_strength": 0.028533049262069288,
    "rsm": 0.41630399524229533,
    "border_count": 167,
    "boosting_type": "Plain",
    "bootstrap_type": "Bernoulli",
    "grow_policy": "Depthwise",
    "min_data_in_leaf": 482,
    "score_function": "L2",
    "subsample": 0.8907108810335598,
    "random_state": 42,
    "thread_count": n_cores,
    "verbose": False,
    "objective": "RMSE",
    "cat_features": [
        "home_ownership",
        "verification_status",
        "purpose",
        "addr_state",
        "initial_list_status",
        "application_type",
        "disbursement_method",
    ],
}
histgbm_full_reg_params = {
    "max_iter": 6790,
    "learning_rate": 0.0018532518685825527,
    "max_depth": 15,
    "min_samples_leaf": 362,
    "max_features": 0.4243152639220175,
    "max_leaf_nodes": 206,
    "l2_regularization": 0.06879760344829619,
    "max_bins": 213,
    "random_state": 42,
    "verbose": 0,
}
pgbm_full_reg_params = {
    "max_iter": 5110,
    "learning_rate": 0.001967232115519223,
    "max_depth": 14,
    "min_samples_leaf": 296,
    "max_leaf_nodes": 192,
    "l2_regularization": 0.38777665330714267,
    "max_bins": 185,
    "tree_correlation": 0.0010571351786840086,
    "distribution": "logistic",
    "random_state": 42,
    "verbose": 0,
    "with_variance": True,
}

models_tuned_full_reg = {
    "XGBoost": (passthrough_preprocessor, XGBRegressor(**xgboost_full_reg_params)),
    "LightGBM": (passthrough_preprocessor, LGBMRegressor(**lightgbm_full_reg_params)),
    "CatBoost": (catboost_preprocessor, CatBoostRegressor(**catboost_full_reg_params)),
    "HistGBM": (numeric_preprocessor_reg, HistGBMRegressor(**histgbm_full_reg_params)),
    "PGBM": (numeric_preprocessor_reg, PGBMRegressor(**pgbm_full_reg_params)),
    "Dummy - Mean": (numeric_preprocessor_reg, DummyRegressor(strategy="mean")),
}

### Evaluation function

In [18]:
import time
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from pathlib import Path
import warnings

# Suppress warnings caused by unseen categories in the test set.
warnings.filterwarnings(
    "ignore",
    message="Found unknown categories in columns.*",
    category=UserWarning,
    module="sklearn.preprocessing._encoders",
)

# Suppress scikit-learn future warning.
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="sklearn.*",
)

# Function to evaluate models on test and train sets, compute metrics, and store predictions
def compare_models_reg(
    models_dict, train_X, train_y, test_X, test_y, dataset_size_name, tuning_status
):
    # Prepare output directories
    out_dir_scores = Path("test_results/Regression/scores")
    out_dir_preds = Path("test_results/Regression/predictions")

    # Create output variables
    results_scores_test = []
    results_scores_train = []

    test_preds_df = pd.DataFrame(index=test_X.index)
    test_preds_df["true_y"] = test_y
    train_preds_df = pd.DataFrame(index=train_X.index)
    train_preds_df["true_y"] = train_y

    # Evaluate each model
    for model_name, (preprocessor, model) in models_dict.items():
        print(
            f"Evaluating {model_name} on {dataset_size_name} dataset ({tuning_status})..."
        )
        pipe = Pipeline([("preprocessor", preprocessor), ("regressor", model)])

        # Fit the model
        start_time_fit = time.time()
        pipe.fit(train_X, train_y)
        fit_time = time.time() - start_time_fit

        # Get number of trees to compute time per tree
        actual_model = pipe.named_steps["regressor"]
        n_trees = get_number_of_trees(model_name, actual_model)
        n_trees = n_trees if n_trees is not None else 1

        # Predict and compute metrics for test set
        start_time_pred = time.time()
        test_pred = pipe.predict(test_X)
        test_pred_time = time.time() - start_time_pred
        test_rmse = np.sqrt(mean_squared_error(test_y, test_pred))
        test_r2 = r2_score(test_y, test_pred)
        test_mae = mean_absolute_error(test_y, test_pred)

        # Predict and compute metrics for train set
        start_time_pred_train = time.time()
        train_pred = pipe.predict(train_X)
        train_pred_time = time.time() - start_time_pred_train
        train_rmse = np.sqrt(mean_squared_error(train_y, train_pred))
        train_r2 = r2_score(train_y, train_pred)
        train_mae = mean_absolute_error(train_y, train_pred)

        # Save predictions and results
        test_preds_df[model_name] = test_pred
        train_preds_df[model_name] = train_pred
        fit_time_per_tree = fit_time / n_trees * 1000
        test_pred_time_per_tree = test_pred_time / n_trees * 1000
        train_pred_time_per_tree = train_pred_time / n_trees * 1000

        results_scores_test.append(
            {
                "Dataset": dataset_size_name,
                "Model": model_name,
                "RMSE": test_rmse,
                "R2": test_r2,
                "MAE": test_mae,
                "Fit Time (s)": fit_time,
                "Pred Time (s)": test_pred_time,
                "Fit Time (ms/tree)": fit_time_per_tree,
                "Pred Time (ms/tree)": test_pred_time_per_tree,
                "Trees": n_trees,
            }
        )
        results_scores_train.append(
            {
                "Dataset": dataset_size_name,
                "Model": model_name,
                "RMSE": train_rmse,
                "R2": train_r2,
                "MAE": train_mae,
                "Fit Time (s)": fit_time,
                "Pred Time (s)": train_pred_time,
                "Fit Time (ms/tree)": fit_time_per_tree,
                "Pred Time (ms/tree)": train_pred_time_per_tree,
                "Trees": n_trees,
            }
        )

    # Convert results to DataFrames
    df_scores_test = pd.DataFrame(results_scores_test)
    df_scores_train = pd.DataFrame(results_scores_train)

    # Save predictions and scores to files
    test_preds_path = (
        out_dir_preds / f"test_preds_{dataset_size_name}_{tuning_status}.parquet"
    )
    train_preds_path = (
        out_dir_preds / f"train_preds_{dataset_size_name}_{tuning_status}.parquet"
    )
    test_scores_path = (
        out_dir_scores / f"test_scores_{dataset_size_name}_{tuning_status}.csv"
    )
    train_scores_path = (
        out_dir_scores / f"train_scores_{dataset_size_name}_{tuning_status}.csv"
    )

    test_preds_df.to_parquet(test_preds_path)
    train_preds_df.to_parquet(train_preds_path)
    df_scores_test.to_csv(test_scores_path, index=False)
    df_scores_train.to_csv(train_scores_path, index=False)

    # Show results
    print(
        f"Finished evaluating {dataset_size_name} dataset ({tuning_status}). Results saved to {out_dir_scores}/ and {out_dir_preds}/."
    )
    display(df_scores_train)
    display(df_scores_test)

### Testing

In [20]:
compare_models_reg(models_no_tuning_reg, train_1k_X, train_1k_y_reg, test_full_X, test_full_y_reg, "1k", "not_tuned")

Evaluating XGBoost on 1k dataset (not_tuned)...
Evaluating LightGBM on 1k dataset (not_tuned)...
Evaluating CatBoost on 1k dataset (not_tuned)...
Evaluating NGBoost on 1k dataset (not_tuned)...
Evaluating GBM on 1k dataset (not_tuned)...
Evaluating HistGBM on 1k dataset (not_tuned)...
Evaluating PGBM on 1k dataset (not_tuned)...
Evaluating Dummy - Mean on 1k dataset (not_tuned)...
Finished evaluating 1k dataset (not_tuned). Results saved to test_results/Regression/scores/ and test_results/Regression/predictions/.


,Dataset,Model,RMSE,R2,MAE,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,1k,XGBoost,0.001461,0.999979,0.000966,0.227280,0.002963,2.272801,0.029631,100
1,1k,LightGBM,0.048307,0.977080,0.033001,0.409950,0.004226,4.099503,0.042260,100
2,1k,CatBoost,0.088798,0.922552,0.063254,1.983135,0.005504,1.983135,0.005504,1000
3,1k,NGBoost,0.236882,0.448856,0.168950,8.767177,0.130991,17.534354,0.261982,500
4,1k,GBM,0.209093,0.570584,0.148497,0.720305,0.008078,7.203047,0.080779,100
5,1k,HistGBM,0.067355,0.955440,0.044577,0.187789,0.006584,1.877892,0.065837,100
6,1k,PGBM,0.063231,0.960730,0.042975,0.474810,0.007135,4.748099,0.071352,100
7,1k,Dummy - Mean,0.319080,0.000000,0.233678,0.005230,0.003226,5.229712,3.226042,1


,Dataset,Model,RMSE,R2,MAE,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,1k,XGBoost,0.387414,-0.129695,0.263514,0.227280,0.159620,2.272801,1.596203,100
1,1k,LightGBM,0.377219,-0.071020,0.258389,0.409950,0.432222,4.099503,4.322224,100
2,1k,CatBoost,0.367630,-0.017263,0.255020,1.983135,0.376144,1.983135,0.376144,1000
3,1k,NGBoost,0.367445,-0.016241,0.254156,8.767177,20.890972,17.534354,41.781944,500
4,1k,GBM,0.370503,-0.033225,0.255724,0.720305,1.188133,7.203047,11.881332,100
5,1k,HistGBM,0.376091,-0.064627,0.258127,0.187789,0.815032,1.877892,8.150320,100
6,1k,PGBM,0.377182,-0.070812,0.257268,0.474810,0.820379,4.748099,8.203788,100
7,1k,Dummy - Mean,0.365444,-0.005199,0.262596,0.005230,0.295871,5.229712,295.870781,1


In [21]:
compare_models_reg(models_no_tuning_reg, train_10k_X, train_10k_y_reg, test_full_X, test_full_y_reg, "10k", "not_tuned")

Evaluating XGBoost on 10k dataset (not_tuned)...
Evaluating LightGBM on 10k dataset (not_tuned)...
Evaluating CatBoost on 10k dataset (not_tuned)...
Evaluating NGBoost on 10k dataset (not_tuned)...
Evaluating GBM on 10k dataset (not_tuned)...
Evaluating HistGBM on 10k dataset (not_tuned)...
Evaluating PGBM on 10k dataset (not_tuned)...
Evaluating Dummy - Mean on 10k dataset (not_tuned)...
Finished evaluating 10k dataset (not_tuned). Results saved to test_results/Regression/scores/ and test_results/Regression/predictions/.


,Dataset,Model,RMSE,R2,MAE,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,10k,XGBoost,0.108776,0.866734,0.073763,0.315415,0.008336,3.154149,0.083358,100
1,10k,LightGBM,0.210749,0.499755,0.145612,0.453831,0.014169,4.538310,0.141690,100
2,10k,CatBoost,0.211190,0.497662,0.146262,3.831685,0.018398,3.831685,0.018398,1000
3,10k,NGBoost,0.280900,0.111302,0.196717,73.683023,0.671535,147.366046,1.343070,500
4,10k,GBM,0.277429,0.133135,0.192751,6.807387,0.040758,68.073871,0.407581,100
5,10k,HistGBM,0.222099,0.444426,0.153120,0.282166,0.021714,2.821658,0.217140,100
6,10k,PGBM,0.220517,0.452314,0.152488,0.585752,0.020393,5.857522,0.203929,100
7,10k,Dummy - Mean,0.297972,0.000000,0.214749,0.013965,0.010024,13.964653,10.024309,1


,Dataset,Model,RMSE,R2,MAE,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,10k,XGBoost,0.375599,-0.061844,0.258467,0.315415,0.153169,3.154149,1.531689,100
1,10k,LightGBM,0.361430,0.016759,0.246587,0.453831,0.237361,4.538310,2.373610,100
2,10k,CatBoost,0.359669,0.026316,0.247068,3.831685,0.352375,3.831685,0.352375,1000
3,10k,NGBoost,0.359745,0.025909,0.247892,73.683023,20.724943,147.366046,41.449885,500
4,10k,GBM,0.360497,0.021828,0.246299,6.807387,1.163096,68.073871,11.630960,100
5,10k,HistGBM,0.360990,0.019152,0.246089,0.282166,0.586285,2.821658,5.862849,100
6,10k,PGBM,0.361257,0.017698,0.246729,0.585752,0.536056,5.857522,5.360560,100
7,10k,Dummy - Mean,0.365843,-0.007397,0.259692,0.013965,0.291168,13.964653,291.167974,1


In [22]:
compare_models_reg(models_no_tuning_reg, train_100k_X, train_100k_y_reg, test_full_X, test_full_y_reg, "100k", "not_tuned")

Evaluating XGBoost on 100k dataset (not_tuned)...
Evaluating LightGBM on 100k dataset (not_tuned)...
Evaluating CatBoost on 100k dataset (not_tuned)...
Evaluating NGBoost on 100k dataset (not_tuned)...
Evaluating GBM on 100k dataset (not_tuned)...
Evaluating HistGBM on 100k dataset (not_tuned)...
Evaluating PGBM on 100k dataset (not_tuned)...
Evaluating Dummy - Mean on 100k dataset (not_tuned)...
Finished evaluating 100k dataset (not_tuned). Results saved to test_results/Regression/scores/ and test_results/Regression/predictions/.


,Dataset,Model,RMSE,R2,MAE,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,100k,XGBoost,0.252317,0.318660,0.174530,1.110657,0.067431,11.106567,0.674310,100
1,100k,LightGBM,0.281411,0.152473,0.196882,0.920518,0.085601,9.205177,0.856011,100
2,100k,CatBoost,0.275267,0.189078,0.192215,19.528042,0.139431,19.528042,0.139431,1000
3,100k,NGBoost,0.295535,0.065265,0.207900,754.539406,7.935683,1509.078812,15.871366,500
4,100k,GBM,0.294957,0.068915,0.207247,70.443365,0.416026,704.433651,4.160261,100
5,100k,HistGBM,0.287802,0.113541,0.201741,1.138547,0.153289,11.385469,1.532891,100
6,100k,PGBM,0.287447,0.115730,0.201434,1.800774,0.142466,18.007739,1.424661,100
7,100k,Dummy - Mean,0.305678,0.000000,0.224233,0.100204,0.075480,100.204229,75.479984,1


,Dataset,Model,RMSE,R2,MAE,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,100k,XGBoost,0.359005,0.029909,0.248358,1.110657,0.210811,11.106567,2.108109,100
1,100k,LightGBM,0.354212,0.055642,0.247915,0.920518,0.215721,9.205177,2.157209,100
2,100k,CatBoost,0.353594,0.058934,0.245919,19.528042,0.358019,19.528042,0.358019,1000
3,100k,NGBoost,0.355452,0.049018,0.251318,754.539406,21.261543,1509.078812,42.523086,500
4,100k,GBM,0.355266,0.050013,0.250529,70.443365,1.185408,704.433651,11.854081,100
5,100k,HistGBM,0.354523,0.053983,0.249295,1.138547,0.431158,11.385469,4.311581,100
6,100k,PGBM,0.354673,0.053182,0.249103,1.800774,0.419827,18.007739,4.198270,100
7,100k,Dummy - Mean,0.364812,-0.001725,0.268982,0.100204,0.239316,100.204229,239.315987,1


In [19]:
compare_models_reg(models_no_tuning_reg, train_full_X, train_full_y_reg, test_full_X, test_full_y_reg, "full", "not_tuned")

Evaluating XGBoost on full dataset (not_tuned)...
Evaluating LightGBM on full dataset (not_tuned)...
Evaluating CatBoost on full dataset (not_tuned)...
Evaluating NGBoost on full dataset (not_tuned)...
Evaluating GBM on full dataset (not_tuned)...
Evaluating HistGBM on full dataset (not_tuned)...
Evaluating PGBM on full dataset (not_tuned)...
Evaluating Dummy - Mean on full dataset (not_tuned)...
Finished evaluating full dataset (not_tuned). Results saved to test_results/Regression/scores/ and test_results/Regression/predictions/.


,Dataset,Model,RMSE,R2,MAE,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,full,XGBoost,0.233315,0.108163,0.139528,9.145659,0.650735,91.456592,6.507351,100
1,full,LightGBM,0.238476,0.068270,0.143137,5.236302,1.239503,52.363019,12.395031,100
2,full,CatBoost,0.235457,0.091711,0.140859,200.569649,1.608687,200.569649,1.608687,1000
3,full,NGBoost,0.241335,0.045798,0.145959,7822.400559,84.318046,15644.801118,168.636092,500
4,full,GBM,0.240986,0.048558,0.145499,704.193945,6.548101,7041.939452,65.481012,100
5,full,HistGBM,0.238833,0.065483,0.143269,18.992773,2.209708,189.927733,22.097080,100
6,full,PGBM,0.238802,0.065723,0.143221,20.937206,1.963037,209.372060,19.630373,100
7,full,Dummy - Mean,0.247059,0.000000,0.154984,1.024746,0.804274,1024.745941,804.273844,1


,Dataset,Model,RMSE,R2,MAE,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,full,XGBoost,0.355816,0.047066,0.243955,9.145659,0.157638,91.456592,1.576381,100
1,full,LightGBM,0.355347,0.049580,0.246876,5.236302,0.304965,52.363019,3.049650,100
2,full,CatBoost,0.354342,0.054947,0.242652,200.569649,0.381504,200.569649,0.381504,1000
3,full,NGBoost,0.358315,0.033637,0.249786,7822.400559,21.754277,15644.801118,43.508554,500
4,full,GBM,0.357994,0.035365,0.249279,704.193945,1.244452,7041.939452,12.444518,100
5,full,HistGBM,0.355620,0.048115,0.247086,18.992773,0.611992,189.927733,6.119921,100
6,full,PGBM,0.355565,0.048414,0.247120,20.937206,0.533629,209.372060,5.336292,100
7,full,Dummy - Mean,0.368242,-0.020653,0.247643,1.024746,0.248421,1024.745941,248.420954,1


In [23]:
compare_models_reg(models_tuned_1k_reg, train_1k_X, train_1k_y_reg, test_full_X, test_full_y_reg, "1k", "tuned")

Evaluating XGBoost on 1k dataset (tuned)...
Evaluating LightGBM on 1k dataset (tuned)...
Evaluating CatBoost on 1k dataset (tuned)...
Evaluating NGBoost on 1k dataset (tuned)...
Evaluating GBM on 1k dataset (tuned)...
Evaluating HistGBM on 1k dataset (tuned)...
Evaluating PGBM on 1k dataset (tuned)...
Evaluating Dummy - Mean on 1k dataset (tuned)...
Finished evaluating 1k dataset (tuned). Results saved to test_results/Regression/scores/ and test_results/Regression/predictions/.


,Dataset,Model,RMSE,R2,MAE,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,1k,XGBoost,0.266265,0.303649,0.192357,0.382412,0.003548,0.712127,0.006607,537
1,1k,LightGBM,0.287985,0.185406,0.205657,0.066426,0.002625,0.461292,0.018229,144
2,1k,CatBoost,0.284326,0.205975,0.203134,1.762431,0.005059,1.805769,0.005183,976
3,1k,NGBoost,0.294183,0.149968,0.208448,1.031239,0.059699,3.921060,0.226993,263
4,1k,GBM,0.303723,0.093941,0.216860,0.317052,0.012260,0.326521,0.012626,971
5,1k,HistGBM,0.288199,0.184193,0.204389,0.166346,0.008910,0.226629,0.012139,734
6,1k,PGBM,0.304042,0.092036,0.219924,0.120050,0.006470,0.451317,0.024323,266
7,1k,Dummy - Mean,0.319080,0.000000,0.233678,0.005195,0.003261,5.194902,3.260851,1


,Dataset,Model,RMSE,R2,MAE,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,1k,XGBoost,0.361989,0.013717,0.252485,0.382412,0.308321,0.712127,0.574155,537
1,1k,LightGBM,0.362153,0.012822,0.251952,0.066426,0.195638,0.461292,1.358598,144
2,1k,CatBoost,0.361271,0.017627,0.251841,1.762431,0.295541,1.805769,0.302808,976
3,1k,NGBoost,0.362436,0.011280,0.249194,1.031239,7.063350,3.921060,26.856844,263
4,1k,GBM,0.363076,0.007781,0.251366,0.317052,3.177834,0.326521,3.272744,971
5,1k,HistGBM,0.362832,0.009117,0.250922,0.166346,0.814028,0.226629,1.109030,734
6,1k,PGBM,0.363664,0.004570,0.257664,0.120050,0.405433,0.451317,1.524185,266
7,1k,Dummy - Mean,0.365444,-0.005199,0.262596,0.005195,0.292345,5.194902,292.345047,1


In [24]:
compare_models_reg(models_tuned_10k_reg, train_10k_X, train_10k_y_reg, test_full_X, test_full_y_reg, "10k", "tuned")

Evaluating XGBoost on 10k dataset (tuned)...
Evaluating LightGBM on 10k dataset (tuned)...
Evaluating CatBoost on 10k dataset (tuned)...
Evaluating NGBoost on 10k dataset (tuned)...
Evaluating GBM on 10k dataset (tuned)...
Evaluating HistGBM on 10k dataset (tuned)...
Evaluating PGBM on 10k dataset (tuned)...
Evaluating Dummy - Mean on 10k dataset (tuned)...
Finished evaluating 10k dataset (tuned). Results saved to test_results/Regression/scores/ and test_results/Regression/predictions/.


,Dataset,Model,RMSE,R2,MAE,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,10k,XGBoost,0.283283,0.096161,0.197974,2.203914,0.046482,1.107494,0.023358,1990
1,10k,LightGBM,0.285964,0.078975,0.199358,1.140495,0.105473,0.763383,0.070598,1494
2,10k,CatBoost,0.280234,0.115517,0.196096,7.509982,0.017582,6.902557,0.016160,1088
3,10k,NGBoost,0.278178,0.128447,0.193186,21.480711,1.664076,15.042515,1.165319,1428
4,10k,GBM,0.291648,0.041993,0.203302,13.866300,0.080994,7.391418,0.043174,1876
5,10k,HistGBM,0.289052,0.058972,0.203132,1.107096,0.054826,1.260929,0.062444,878
6,10k,PGBM,0.288585,0.062012,0.202014,1.568921,0.046620,2.341673,0.069582,670
7,10k,Dummy - Mean,0.297972,0.000000,0.214749,0.014438,0.009952,14.438152,9.952068,1


,Dataset,Model,RMSE,R2,MAE,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,10k,XGBoost,0.358604,0.032073,0.247117,2.203914,1.154431,1.107494,0.580116,1990
1,10k,LightGBM,0.358910,0.030423,0.246897,1.140495,2.819380,0.763383,1.887135,1494
2,10k,CatBoost,0.359003,0.029923,0.247574,7.509982,0.343122,6.902557,0.315369,1088
3,10k,NGBoost,0.358328,0.033563,0.245811,21.480711,47.257182,15.042515,33.093265,1428
4,10k,GBM,0.360097,0.023999,0.247332,13.866300,2.609749,7.391418,1.391124,1876
5,10k,HistGBM,0.360159,0.023666,0.249443,1.107096,1.278329,1.260929,1.455956,878
6,10k,PGBM,0.360247,0.023187,0.248402,1.568921,1.148456,2.341673,1.714113,670
7,10k,Dummy - Mean,0.365843,-0.007397,0.259692,0.014438,0.306139,14.438152,306.139231,1


In [25]:
compare_models_reg(models_tuned_100k_reg, train_100k_X, train_100k_y_reg, test_full_X, test_full_y_reg, "100k", "tuned")

Evaluating XGBoost on 100k dataset (tuned)...
Evaluating LightGBM on 100k dataset (tuned)...
Evaluating CatBoost on 100k dataset (tuned)...
Evaluating NGBoost on 100k dataset (tuned)...
Evaluating GBM on 100k dataset (tuned)...
Evaluating HistGBM on 100k dataset (tuned)...
Evaluating PGBM on 100k dataset (tuned)...
Evaluating Dummy - Mean on 100k dataset (tuned)...
Finished evaluating 100k dataset (tuned). Results saved to test_results/Regression/scores/ and test_results/Regression/predictions/.


,Dataset,Model,RMSE,R2,MAE,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,100k,XGBoost,0.275546,0.187437,0.192718,45.360407,3.997394,4.909135,0.432618,9240
1,100k,LightGBM,0.283058,0.142523,0.197841,56.757245,18.486177,8.420956,2.742756,6740
2,100k,CatBoost,0.284365,0.134589,0.198957,294.864792,3.190367,29.694340,0.321286,9930
3,100k,NGBoost,0.294271,0.073241,0.206088,828.805006,49.510203,234.125708,13.985933,3540
4,100k,GBM,0.294143,0.074048,0.205881,2029.553490,5.179011,337.135131,0.860301,6020
5,100k,HistGBM,0.290319,0.097967,0.203630,47.344657,4.632204,9.863470,0.965043,4800
6,100k,PGBM,0.291246,0.092201,0.203736,16.756743,0.975809,1.765726,0.102825,9490
7,100k,Dummy - Mean,0.305678,0.000000,0.224233,0.107288,0.079671,107.288122,79.670906,1


,Dataset,Model,RMSE,R2,MAE,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,100k,XGBoost,0.353091,0.061606,0.246703,45.360407,10.768795,4.909135,1.165454,9240
1,100k,LightGBM,0.353349,0.060236,0.247216,56.757245,51.794640,8.420956,7.684665,6740
2,100k,CatBoost,0.353326,0.060360,0.247658,294.864792,8.520876,29.694340,0.858094,9930
3,100k,NGBoost,0.354432,0.054468,0.249150,828.805006,135.033440,234.125708,38.145040,3540
4,100k,GBM,0.354122,0.056120,0.248226,2029.553490,14.626043,337.135131,2.429575,6020
5,100k,HistGBM,0.354229,0.055550,0.249305,47.344657,12.680125,9.863470,2.641693,4800
6,100k,PGBM,0.354149,0.055974,0.248741,16.756743,2.656330,1.765726,0.279908,9490
7,100k,Dummy - Mean,0.364812,-0.001725,0.268982,0.107288,0.262917,107.288122,262.917280,1


In [20]:
compare_models_reg(models_tuned_full_reg, train_full_X, train_full_y_reg, test_full_X, test_full_y_reg, "full", "tuned")

Evaluating XGBoost on full dataset (tuned)...
Evaluating LightGBM on full dataset (tuned)...
Evaluating CatBoost on full dataset (tuned)...
Evaluating HistGBM on full dataset (tuned)...
Evaluating PGBM on full dataset (tuned)...
Evaluating Dummy - Mean on full dataset (tuned)...
Finished evaluating full dataset (tuned). Results saved to test_results/Regression/scores/ and test_results/Regression/predictions/.


,Dataset,Model,RMSE,R2,MAE,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,full,XGBoost,0.226624,0.158579,0.136158,786.558793,126.370378,115.670411,18.583879,6800
1,full,LightGBM,0.229508,0.137031,0.136968,526.086671,446.031697,88.566780,75.089511,5940
2,full,CatBoost,0.231530,0.121755,0.138578,4430.060385,31.668696,504.562686,3.606913,8780
3,full,HistGBM,0.232986,0.110673,0.139171,1637.355070,165.325210,241.142131,24.348337,6790
4,full,PGBM,0.233396,0.107548,0.139468,1512.170079,119.923110,295.923695,23.468319,5110
5,full,Dummy - Mean,0.247059,0.000000,0.154984,1.066702,0.798085,1066.701889,798.084974,1


,Dataset,Model,RMSE,R2,MAE,Fit Time (s),Pred Time (s),Fit Time (ms/tree),Pred Time (ms/tree),Trees
0,full,XGBoost,0.353736,0.058175,0.245124,786.558793,28.638180,115.670411,4.211497,6800
1,full,LightGBM,0.353352,0.060218,0.242377,526.086671,108.877959,88.566780,18.329623,5940
2,full,CatBoost,0.353619,0.058798,0.245271,4430.060385,7.831913,504.562686,0.892017,8780
3,full,HistGBM,0.353881,0.057403,0.243983,1637.355070,42.520013,241.142131,6.262152,6790
4,full,PGBM,0.354182,0.055801,0.244177,1512.170079,30.995104,295.923695,6.065578,5110
5,full,Dummy - Mean,0.368242,-0.020653,0.247643,1.066702,0.243141,1066.701889,243.140936,1
